# Chapter 18 — Claims, Evidence, and Decisions

**Companion to *Applied AI*.**

A review says three things about a cache change, and someone merges it.
The next day a deployed configuration disagrees with one of them.
This notebook replays that preserved run: which statements were supported,
by what, and what happened to the recorded decision when its basis moved.

## Question

**What happens to a recorded decision when its basis moves?**

## What this notebook does

It **inspects** the preserved Stage 18 bundle
(`claims-evidence/2026-09-14-3b6d8fb/`, case `decision-over-days`):
re-derives the day-1 standings, shows the recorded decision and its snapshot,
then replays day 2 — one refuting source — and names exactly what changed
while the decision record stays byte-identical.

```text
claim  !=  evidence  !=  decision  !=  standing
```

## Setup

Standard library only. No network, no API key, no `codeai` import.
The evidence directory is located the same way in every notebook and can be
overridden with `APPLIED_AI_EVIDENCE`; only bundle-relative paths are shown.

In [1]:
import json
import os
from pathlib import Path

def find_evidence_dir(marker="claims-evidence"):
    """Locate the preserved evidence. Override with APPLIED_AI_EVIDENCE."""
    env = os.environ.get("APPLIED_AI_EVIDENCE")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "evidence",
                     base / "experiments" / "applied-ai" / "evidence"):
            if (cand / marker).is_dir():
                return cand
    raise FileNotFoundError(
        "Preserved evidence not found. Set APPLIED_AI_EVIDENCE to the "
        "directory holding the Applied AI evidence bundles.")

EVIDENCE_DIR = find_evidence_dir()
CASE = EVIDENCE_DIR / "claims-evidence" / "2026-09-14-3b6d8fb" / "decision-over-days"
print("bundle: claims-evidence/2026-09-14-3b6d8fb/decision-over-days")
print("files :", sorted(p.name for p in CASE.iterdir() if p.is_file())[:6], "...")

decided = json.loads((CASE / "inspect-decided.json").read_text(encoding="utf-8"))
day2 = json.loads((CASE / "inspect-day2.json").read_text(encoding="utf-8"))
events_decided = json.loads((CASE / "events-decided.json").read_text(encoding="utf-8"))
events_day2 = json.loads((CASE / "events-day2.json").read_text(encoding="utf-8"))
print("day-1 events:", len(events_decided), "| day-2 events:", len(events_day2))

bundle: claims-evidence/2026-09-14-3b6d8fb/decision-over-days
files : ['action-contradict.json', 'action-decide.json', 'action-evidence.json', 'action-extract.json', 'events-attributed.json', 'events-day2.json'] ...
day-1 events: 27 | day-2 events: 28


## 1. Said is not supported

Day 1, three claims were extracted as exact spans of the preserved review —
characters 0–56 (TTL), 57–79 (retry test), 80–111 (latency).
Every one of them started **unresolved at E1_ATTRIBUTED**, whatever it said.
A fourth extraction, quoting the review as saying "600 seconds", did not match
the preserved text at its span and was refused (`quote_not_in_source`).

Support came later, and separately:

In [2]:
print(f"{'claim':<10}{'quote':<55}{'status':<12}{'class':<16}evidence")
print("-" * 110)
for cid in ("c-ttl", "c-retry", "c-latency"):
    s = decided["standings"][cid]
    print(f"{cid:<10}{s['quote']:<55}{s['status']:<12}"
          f"{s['evidence_class']:<16}{s['supporting_evidence_ids']}")

st = decided["standings"]
assert st["c-ttl"]["status"] == "supported" and st["c-ttl"]["evidence_class"] == "E2_SOURCE_CHECKED"
assert st["c-ttl"]["supporting_evidence_ids"] == ["ev-config"]       # ttl_seconds = 60, human reviewer
assert st["c-retry"]["status"] == "supported" and st["c-retry"]["evidence_class"] == "E3_REPRODUCED"
assert st["c-latency"]["status"] == "unresolved" and st["c-latency"]["evidence_class"] == "E1_ATTRIBUTED"
print()
print("assertion held: E2 via a source passage, E3 via a completed check that named")
print("the claim, and one claim left honestly unresolved.")

claim     quote                                                  status      class           evidence
--------------------------------------------------------------------------------------------------------------
c-ttl     The cache TTL is set to 60 seconds in config/cache.toml.supported   E2_SOURCE_CHECKED['ev-config']
c-retry   The retry test passes.                                 supported   E3_REPRODUCED   ['ev-retry']
c-latency p99 latency stays under 200 ms.                        unresolved  E1_ATTRIBUTED   []

assertion held: E2 via a source passage, E3 via a completed check that named
the claim, and one claim left honestly unresolved.


## 2. The decision, with its reasons attached

The first merge attempt — resting on the TTL *and latency* claims — was
refused (`claim_not_supported:c-latency`). The recorded decision,
`dec-merge`, relies on the two supported claims and **acknowledges** the
latency claim as unresolved. It stores a snapshot of each relied-on claim:
status, evidence class, evidence IDs for and against, quote and observation
hashes, and the source call's adopted status.

In [3]:
dec = decided["decisions"]["dec-merge"]
print("decision      :", dec["decision_id"])
print("relied on     :", dec["relied_on_claim_ids"])
print("standing      :", dec["standing"])
print("changes       :", dec["changes"])
print("resting on c-ttl    :", decided["resting_on"]["c-ttl"])
print("resting on c-retry  :", decided["resting_on"]["c-retry"])
print("resting on c-latency:", decided["resting_on"]["c-latency"], "(acknowledged, not relied on)")
assert dec["standing"] == "basis_intact" and dec["changes"] == []
assert decided["resting_on"]["c-latency"] == []
print()
print("assertion held: the merge is recorded on a fully supported basis,")
print("with the open claim named as open rather than silently relied on.")

decision      : dec-merge
relied on     : ['c-ttl', 'c-retry']
standing      : basis_intact
changes       : []
resting on c-ttl    : ['dec-merge']
resting on c-retry  : ['dec-merge']
resting on c-latency: [] (acknowledged, not relied on)

assertion held: the merge is recorded on a fully supported basis,
with the open claim named as open rather than silently relied on.


## 3. Day two: one refuting source

A new process recorded one more piece of evidence. An on-call engineer cited
`ttl_seconds = 600` in the **deployed** configuration — a different preserved
file — as refuting the TTL claim. The claim keeps its E2 support and gains a
refutation. CodeAI does not choose between them: the claim becomes
**contested**.

In [4]:
ev = next(e for e in events_day2 if e.get("kind") == "claim.evidence_recorded"
            and e["payload"].get("evidence_id") == "ev-deployed")
p = ev["payload"]
print("evidence :", p["evidence_id"], "| claim:", p["claim_id"], "| recorder:", p["actor_id"])
print("passage  :", repr(p["passage"]), "| kind:", p["kind"])
print("c-ttl status now:", day2["standings"]["c-ttl"]["status"],
      "| supporting:", day2["standings"]["c-ttl"]["supporting_evidence_ids"],
      "| refuting:", day2["standings"]["c-ttl"]["refuting_evidence_ids"])

d2 = day2["decisions"]["dec-merge"]
print()
print("decision standing:", d2["standing"])
for ch in d2["changes"]:
    print(f"  {ch['claim_id']} . {ch['field']}: {ch['recorded']!r} -> {ch['current']!r}")

assert day2["standings"]["c-ttl"]["status"] == "contested"
assert d2["standing"] == "basis_changed"
assert d2["changes"] == [
    {"claim_id": "c-ttl", "field": "status", "recorded": "supported", "current": "contested"},
    {"claim_id": "c-ttl", "field": "refuting_evidence_ids", "recorded": [], "current": ["ev-deployed"]},
]
print()
print("assertion held: exactly two named changes, both on c-ttl; the retry entry did not move.")

evidence : ev-deployed | claim: c-ttl | recorder: on-call-engineer
passage  : 'ttl_seconds = 600' | kind: source_passage
c-ttl status now: contested | supporting: ['ev-config'] | refuting: ['ev-deployed']

decision standing: basis_changed
  c-ttl . status: 'supported' -> 'contested'
  c-ttl . refuting_evidence_ids: [] -> ['ev-deployed']

assertion held: exactly two named changes, both on c-ttl; the retry entry did not move.


## 4. The record did not move

The decision's standing changed. The decision itself did not: the
`decision.recorded` event on day 2 is byte-for-byte the event recorded on
day 1, and so is every event before it. History stays historical; only the
projection of *current* standing moves.

In [5]:
DID = "0f0a407f-4c93-4880-9038-d03817f78997"
print("decision_event_id, day 1:", decided["decisions"]["dec-merge"]["decision_event_id"])
print("decision_event_id, day 2:", day2["decisions"]["dec-merge"]["decision_event_id"])
assert decided["decisions"]["dec-merge"]["decision_event_id"] == DID == day2["decisions"]["dec-merge"]["decision_event_id"]

before = next(e for e in events_decided if e.get("event_id") == DID)
after = next(e for e in events_day2 if e.get("event_id") == DID)
assert json.dumps(before, sort_keys=True) == json.dumps(after, sort_keys=True)
print("byte-identical decision event across both exports:", True)
print("projection events read day 1 / day 2:", decided["events_before"], "/", day2["events_after"])
print()
print("Do not mutate old evidence to manufacture consistency: the merge was made")
print("on a supported basis, and it is shown to have been made on a basis that moved.")

decision_event_id, day 1: 0f0a407f-4c93-4880-9038-d03817f78997
decision_event_id, day 2: 0f0a407f-4c93-4880-9038-d03817f78997
byte-identical decision event across both exports: True
projection events read day 1 / day 2: 27 / 28

Do not mutate old evidence to manufacture consistency: the merge was made
on a supported basis, and it is shown to have been made on a basis that moved.


## 5. Evidence that does not count

The same run refused, each time appending only the refusal:

- **self-evidence** — the reviewing model recording a passage for its own claim;
- **circular evidence** — the human citing the review's own response bytes;
- **an untargeted check** — a TOML parse that named only the TTL claim, cited for the retry claim;
- **agreement** — the same TTL sentence from a second model; both claims stayed
  unresolved at E1, and a decision resting on "two models agree" was refused.

In [6]:
sem = json.loads((EVIDENCE_DIR / "claims-evidence" / "2026-09-14-3b6d8fb"
                   / "verification-semantic.json").read_text(encoding="utf-8"))
ref = next(c for c in sem["claims"] if c["claim_id"] == "C-B:refusals")
print("verifier claim :", ref["description"])
print("verdict        :", ref["verdict"])
print("observed       :")
print(json.dumps(ref["observed"], indent=2)[:900])
assert ref["verdict"] == "PASS"
print()
print("The independent verifier — which imports neither CodeAI nor the producer —")
print("re-derived every refusal reason from the exported events.")

verifier claim : Self-authored, circular and untargeted evidence was refused for the reasons the verifier derives, appending only the refusal
verdict        : PASS
observed       :
{
  "independent": {
    "ev-circular": true,
    "ev-self": true,
    "ev-untargeted": true
  },
  "reasons": {
    "ev-circular": [
      "source_is_claim_origin"
    ],
    "ev-self": [
      "self_evidence"
    ],
    "ev-untargeted": [
      "check_not_targeting_claim"
    ]
  }
}

The independent verifier — which imports neither CodeAI nor the producer —
re-derived every refusal reason from the exported events.


## Interpretation

1. **Said ≠ supported ≠ relied on.** Attribution records who said something and
   where. Support needs a separate evidence record. Reliance needs a decision
   that snapshots its basis.
2. **Derive standing; snapshot declared reliance.** Status comes from the
   evidence records, never from whoever wrote last. The decision pins what it
   relied on, so a later change names exactly what moved.
3. **A moved basis is not a defeated basis.** `basis_changed` fires on any
   tracked difference — including *added* support. Later work (Chapter 18's
   defeat projection) separates "better founded" from "defeated"; this run
   reports the diff and refuses to rewrite the past.
4. **Contested has no adjudication.** Support plus refutation is a stable end
   state: no new decision may rest on the claim, and nothing in the runtime
   settles the contest. Preserved as a limit, not a bug to smooth over.
5. **Agreement is not evidence.** Two unchecked quotations are still unchecked.

## Try it yourself

1. Open `action-decide.json` in the case directory: find the refused first
   attempt and its `claim_not_supported:c-latency` reason.
2. Check `resting_on` for `c-latency` on day 2 — acknowledged claims name no
   decisions. Why is that the correct answer?
3. Read the `source-reinterpreted` case's final inspection: which tracked
   field moved there, and did any evidence change?

*Evidence: `experiments/applied-ai/evidence/claims-evidence/2026-09-14-3b6d8fb/`
(case `decision-over-days`, independent `verify.py`, 17/17 semantic claims pass).
No network, no API key, no `codeai` import.*